# Integrated MLB / NPB / Soccer pipeline

MLB、NPB、サッカーの入力ファイルを同じ処理系で正規化し、試合前特徴量を作成します。存在しない情報は作らず、coverage_report.csvに記録します。

In [ ]:
!pip -q install pandas requests pyarrow
from pathlib import Path
import pandas as pd, numpy as np, glob, os, re, json
ROOT=Path("/content/sports_prediction"); INPUT=ROOT/"input"; OUT=ROOT/"output"
INPUT.mkdir(parents=True,exist_ok=True); OUT.mkdir(parents=True,exist_ok=True)
print("Upload CSV files into:",INPUT)

In [ ]:
def clean_columns(df):
    x=df.copy(); x.columns=[re.sub(r"[^a-z0-9]+","_",str(c).strip().lower()).strip("_") for c in x.columns]; return x
def load_kind(patterns):
    paths=[]
    for p in patterns: paths += glob.glob(str(INPUT/p))
    frames=[]
    for f in sorted(set(paths)):
        try:
            x=clean_columns(pd.read_csv(f)); x["source_file"]=os.path.basename(f); frames.append(x)
        except Exception as e: print("Skipped",f,e)
    return pd.concat(frames,ignore_index=True) if frames else pd.DataFrame()
def make_features(df, league):
    if df.empty: return df
    x=df.copy(); date_col=next((c for c in ["date","gamedate","game_date","datetime","game_datetime"] if c in x),None)
    if not date_col: return pd.DataFrame()
    x["datetime"]=pd.to_datetime(x[date_col],errors="coerce",utc=True)
    aliases={"home_team":["home_team","hometeam","home_team_name","hometeamnameen"],"away_team":["away_team","awayteam","away_team_name","awayteamnameen"],"home_score":["home_score","homescore","fthg"],"away_score":["away_score","awayscore","ftag"]}
    for target,names in aliases.items():
        c=next((c for c in names if c in x),None); x[target]=x[c] if c else np.nan
    x["home_score"]=pd.to_numeric(x["home_score"],errors="coerce"); x["away_score"]=pd.to_numeric(x["away_score"],errors="coerce")
    x=x.dropna(subset=["datetime","home_team","away_team","home_score","away_score"]).copy(); x["league"]=league
    x["match_id"]=x.get("match_id",league+"_"+x.index.astype(str)); x=x.sort_values(["datetime","match_id"]).drop_duplicates("match_id").reset_index(drop=True)
    x["total_score"]=x.home_score+x.away_score; x["home_win"]=(x.home_score>x.away_score).astype(int); x["away_win"]=(x.away_score>x.home_score).astype(int); x["draw"]=(x.home_score==x.away_score).astype(int); x["prediction_cutoff_at"]=x["datetime"]-pd.Timedelta(hours=1)
    for side,team,score,opp,result in [("home","home_team","home_score","away_score","home_win"),("away","away_team","away_score","home_score","away_win")]:
        x[f"{side}_rest_days"]=x.groupby(team)["datetime"].diff().dt.total_seconds()/86400
        for w in [3,5,10,20,30,45,60]:
            x[f"{side}_score_prior_{w}"]=x.groupby(team)[score].transform(lambda s:s.shift().rolling(w,min_periods=1).mean()); x[f"{side}_conceded_prior_{w}"]=x.groupby(team)[opp].transform(lambda s:s.shift().rolling(w,min_periods=1).mean()); x[f"{side}_win_rate_prior_{w}"]=x.groupby(team)[result].transform(lambda s:s.shift().rolling(w,min_periods=1).mean())
    elo={}; he=[]; ae=[]
    for r in x.itertuples():
        eh,ea=elo.get(r.home_team,1500.),elo.get(r.away_team,1500.); he.append(eh); ae.append(ea); exp=1/(1+10**(-((eh+55)-ea)/400)); actual=1 if r.home_score>r.away_score else .5 if r.home_score==r.away_score else 0; elo[r.home_team]=eh+20*(actual-exp); elo[r.away_team]=ea+20*((1-actual)-(1-exp))
    x["home_elo_prior"]=he; x["away_elo_prior"]=ae; x["elo_difference_prior"]=x.home_elo_prior-x.away_elo_prior; return x

In [ ]:
configs={"MLB":["mlb_games_*.csv"],"NPB":["npb*.csv","*npb*.csv"],"SOCCER":["soccer*.csv","football*.csv","*football*data*.csv","openfootball*.csv"]}
coverage=[]
for league,patterns in configs.items():
    raw=load_kind(patterns); feat=make_features(raw,league); path=OUT/(league.lower()+"_pre_match_features.csv")
    if len(feat): feat.to_csv(path,index=False)
    coverage.append({"league":league,"input_rows":len(raw),"output_rows":len(feat),"output_columns":len(feat.columns) if len(feat) else 0,"output_file":str(path) if len(feat) else ""})
    print(league,"input",len(raw),"output",len(feat))
pd.DataFrame(coverage).to_csv(OUT/"coverage_report.csv",index=False); print("Created",OUT/"coverage_report.csv")

## 追加データ
Statcast、個人選手成績、先発、ラインアップ、ブルペン、天候、球場係数、オッズ、怪我、NPB詳細、サッカーイベント等は、許諾済みCSVを`/content/sports_prediction/input/`へ追加すると別途結合できます。存在しない値は自動生成しません。